# NBA Model Training on Colab GPU

**Seamless workflow:**
1. Clone code from GitHub
2. Mount Google Drive for persistence
3. Configure & train
4. Download models/data

---
**Auto-Sized Models:**
The training script auto-detects your GPU (H100, T4, etc.) and sizes models optimally.
Bigger GPU = Bigger models = Better predictions.

---
**Drive structure after training:**
```
/MyDrive/nba_model/
├── data/              # Cached NBA data
├── models/            # Trained models
└── training_config.json  # Model config used
```

In [ ]:
# @title Cell 1: Setup Environment
# @markdown Run this cell first to set up everything

import os
import sys

# 1. Clone repository (change URL to your fork if needed)
REPO_URL = "https://github.com/mrsba/nbatest.git"  # @param {type:"string"}
REPO_NAME = "nbatest"

if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL}
else:
    print(f"Repository already exists, pulling latest...")
    !cd {REPO_NAME} && git pull

os.chdir(REPO_NAME)
print(f"Working directory: {os.getcwd()}")

# 2. Mount Google Drive
from google.colab import drive
print("\nMounting Google Drive...")
drive.mount('/content/drive')

# 3. Create persistent directories in Drive
DRIVE_BASE = "/content/drive/MyDrive/nba_model"
DRIVE_DATA = f"{DRIVE_BASE}/data"
DRIVE_MODELS = f"{DRIVE_BASE}/models"

os.makedirs(DRIVE_DATA, exist_ok=True)
os.makedirs(DRIVE_MODELS, exist_ok=True)
print(f"Drive directories ready:\n  Data: {DRIVE_DATA}\n  Models: {DRIVE_MODELS}")

# 4. Install dependencies
print("\nInstalling dependencies...")
!pip install -q catboost lightgbm xgboost scikit-learn pandas numpy torch tqdm nba_api joblib pyyaml psutil

# 5. Check GPU and show expected model size
import torch
print("\n" + "="*50)
print("GPU DETECTION")
print("="*50)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram:.1f} GB")
    
    # Estimate model size
    if 'H100' in gpu_name or vram > 70:
        model_size = "ULTRA (H100-class)"
    elif 'A100' in gpu_name:
        model_size = "LARGE (A100)" if vram < 50 else "ULTRA (A100-80GB)"
    elif 'L4' in gpu_name or vram > 20:
        model_size = "MEDIUM+ (L4)"
    elif 'T4' in gpu_name or 'V100' in gpu_name:
        model_size = "MEDIUM (T4/V100)"
    else:
        model_size = "SMALL (Unknown GPU)"
    print(f"\nExpected Model Size: {model_size}")
else:
    print("No GPU detected - will use CPU (SMALL model)")
print("="*50)

In [ ]:
# @title Cell 2: Configure Training
# @markdown ---
# @markdown **Training Mode:**
training_mode = "fetch_and_train"  # @param ["fetch_and_train", "use_cached_train", "train_only"]
# @markdown * `fetch_and_train` - Fetch fresh data, then train (first run)
# @markdown * `use_cached_train` - Use cached data from Drive, then train
# @markdown * `train_only` - Skip data fetch, just train (if data already in Drive)
# @markdown ---
# @markdown **Model Size Override (leave as 'auto' for auto-detection):**
model_size_override = "auto"  # @param ["auto", "small", "medium", "large", "ultra"]
# @markdown * `auto` - Detect GPU and size automatically (recommended)
# @markdown * `small` - Force small model (CPU or weak GPU)
# @markdown * `medium` - Force medium model (T4, V100)
# @markdown * `large` - Force large model (A100-40GB)
# @markdown * `ultra` - Force ultra model (H100, A100-80GB)
# @markdown ---
# @markdown **Data Fetching** (ignored if using cached data):
seasons_mode = "recent"  # @param ["recent", "current", "all", "custom"]
custom_seasons_input = ""  # @param {type:"string"}
force_refresh = False  # @param {type:"boolean"}
# @markdown * `recent` - Last 10 seasons (recommended)
# @markdown * `current` - Current season only (fast, ~5 min)
# @markdown * `all` - All NBA seasons since 1946 (slow, ~2 hours)
# @markdown * `custom` - Specify below (e.g., "2023-24,2024-25" or "70-80")

# Store config for use in next cells
config = {
    'training_mode': training_mode,
    'model_size': model_size_override,
    'seasons_mode': seasons_mode,
    'custom_seasons': custom_seasons_input,
    'force_refresh': force_refresh,
    'drive_data': DRIVE_DATA,
    'drive_models': DRIVE_MODELS
}
print("Configuration saved. Proceed to Cell 3.")

In [ ]:
# @title Cell 3: Fetch Data (if needed)

import subprocess
import shutil

# Check if we need to fetch data
need_fetch = config['training_mode'] == 'fetch_and_train'
use_cached = config['training_mode'] in ['use_cached_train', 'train_only']

# Check for cached data
cached_data_exists = os.path.exists(config['drive_data']) and len(os.listdir(config['drive_data'])) > 0

if use_cached and not cached_data_exists:
    print("WARNING: No cached data found in Drive. Switching to fetch_and_train mode.")
    need_fetch = True
    use_cached = False

if need_fetch:
    cmd = [sys.executable, "update_data.py", "--data-dir", config['drive_data']]
    
    mode = config['seasons_mode']
    
    if mode == "recent":
        cmd.append("--all-seasons")
    elif mode == "all":
        cmd.append("--full-scrape")
    elif mode == "current":
        cmd.append("--current-season")
    elif mode == "custom" and config['custom_seasons'].strip():
        seasons = []
        for part in config['custom_seasons'].split(','):
            part = part.strip()
            if '-' in part:
                start, end = part.split('-', 1)
                try:
                    start_idx = int(start.strip())
                    end_idx = int(end.strip())
                    for i in range(start_idx, end_idx + 1):
                        seasons.append(str(i))
                except:
                    pass
            else:
                seasons.append(part)
        
        ALL_SEASONS = [
            '1946-47', '1947-48', '1948-49', '1949-50',
            '1950-51', '1951-52', '1952-53', '1953-54',
            '1954-55', '1955-56', '1956-57', '1957-58',
            '1958-59', '1959-60', '1960-61', '1961-62',
            '1962-63', '1963-64', '1964-65', '1965-66',
            '1966-67', '1967-68', '1968-69', '1969-70',
            '1970-71', '1971-72', '1972-73', '1973-74',
            '1974-75', '1975-76', '1976-77', '1977-78',
            '1978-79', '1979-80', '1980-81', '1981-82',
            '1982-83', '1983-84', '1984-85', '1985-86',
            '1986-87', '1987-88', '1988-89', '1989-90',
            '1990-91', '1991-92', '1992-93', '1993-94',
            '1994-95', '1995-96', '1996-97', '1997-98',
            '1998-99', '1999-00', '2000-01', '2001-02',
            '2002-03', '2003-04', '2004-05', '2005-06',
            '2006-07', '2007-08', '2008-09', '2009-10',
            '2010-11', '2011-12', '2012-13', '2013-14',
            '2014-15', '2015-16', '2016-17', '2017-18',
            '2018-19', '2019-20', '2020-21', '2021-22',
            '2022-23', '2023-24', '2024-25', '2025-26'
        ]
        
        final_seasons = []
        for s in seasons:
            try:
                idx = int(s)
                if 1 <= idx <= len(ALL_SEASONS):
                    final_seasons.append(ALL_SEASONS[idx - 1])
            except:
                if s in ALL_SEASONS:
                    final_seasons.append(s)
        
        for s in final_seasons:
            cmd.extend(["--season", s])
    
    if config['force_refresh']:
        cmd.append("--force")
    
    print(f"Running: {' '.join(cmd)}")
    print("\n" + "="*50)
    print("Fetching NBA data...")
    print("="*50 + "\n")
    
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
    process.wait()
    print("\nData fetch complete!")

elif use_cached:
    print(f"Using cached data from: {config['drive_data']}")
    print(f"Files: {os.listdir(config['drive_data'])}")

else:
    print("Skipping data fetch (train_only mode with existing data)")

In [ ]:
# @title Cell 4: Train Models
# @markdown Training will auto-size models based on your GPU. Output saved to Google Drive.

model_size_arg = config['model_size']
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)
print(f"Data directory: {config['drive_data']}")
print(f"Models directory: {config['drive_models']}")
if model_size_arg != 'auto':
    print(f"Model size override: {model_size_arg}")
else:
    print("Model size: AUTO (will detect GPU)")
print("="*60 + "\n")

!python train.py --data-dir "{config['drive_data']}" --models-dir "{config['drive_models']}" --model-size {model_size_arg}

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)

In [ ]:
# @title Cell 5: Download Results
# @markdown Select what to download to your local machine:

download_models = True   # @param {type:"boolean"}
download_data = False   # @param {type:"boolean"}
download_config = True  # @param {type:"boolean"}

from google.colab import files as dl_files

downloads = []

if download_models:
    models_path = config['drive_models']
    if os.path.exists(models_path) and os.listdir(models_path):
        print("Zipping models...")
        !zip -rq models.zip "{models_path}"
        dl_files.download('models.zip')
        downloads.append('models.zip')
    else:
        print("WARNING: No models found to download")

if download_data:
    data_path = config['drive_data']
    if os.path.exists(data_path) and os.listdir(data_path):
        print("Zipping data...")
        !zip -rq data.zip "{data_path}"
        dl_files.download('data.zip')
        downloads.append('data.zip')
    else:
        print("WARNING: No data found to download")

if download_config:
    config_path = os.path.join(config['drive_models'], 'training_config.json')
    if os.path.exists(config_path):
        dl_files.download(config_path)
        downloads.append('training_config.json')

if downloads:
    print(f"\nDownloaded: {', '.join(downloads)}")
else:
    print("\nNothing selected for download.")